# Machining Sounds

## Part A: Vibrations

*A FAB26 workshop lesson, companion to* Machining Dynamics: Jupyter Notebook Edition *(DOI: forthcoming)*

**Authors:** Tony L. Schmitz and Michael F. Gomez

> *"The sensation of sound is a thing sui generis, not comparable with any of our other sensations."*
> Lord Rayleigh, *The Theory of Sound* (1877)


Machines make sound, and that sound carries information. Every sound a milling machine makes begins as a vibration of some part of the machine, so vibration is where this lesson series begins. The series has three Parts and one destination: by the end of Part C, you will listen to a simulated cut, identify its chatter frequency, and choose a better spindle speed, with a time domain simulation checking every step. Part A contributes the vocabulary the rest depends on: the natural frequency, stiffness, and damping of a structure, the frequency response function that collects them, and the connection between a vibrating structure and the sound it radiates.

A note on mechanics, stated once for the whole series. A few short parameter cells are left visible and are meant to be edited and rerun. Every figure and analysis cell is collapsed beneath the prose that describes it, and can be expanded whenever the implementation is wanted. Each Part stands alone, is written to punctuate a live demonstration, and reads on its own afterward.

The purposes of Part A are to:

- Review the principal developments in the study of sound and vibration, and the origin of the chatter problem.
- Distinguish the three kinds of vibration: free, forced, and self-excited.
- Describe the simplest vibrating system and its natural frequency.
- Explain damping, and why a struck object rings down rather than ringing forever.
- Describe forced vibration, resonance, and the frequency response function (FRF).
- Introduce the tap test, the standard measurement of tool point dynamics.
- Connect vibration to sound, and learn to read the spectrum and the spectrogram.

Each concept is demonstrated interactively: run the cells, listen, adjust a parameter or a slider, and run again. No prior vibrations background is assumed.


### Nomenclature

The following symbols are used consistently throughout the lesson:

| Symbol | Meaning | Units |
|--------|---------|-------|
| $t$ | time | s |
| $f$ | frequency | Hz |
| $m$ | mass | kg |
| $k$ | stiffness | N/m (often quoted in N/µm) |
| $c$ | viscous damping coefficient | N·s/m |
| $\omega_n$ | natural frequency | rad/s |
| $f_n$ | natural frequency, $f_n = \omega_n / 2\pi$ | Hz |
| $\zeta$ | damping ratio | dimensionless |
| $\omega_d$ | damped natural frequency | rad/s |
| $r$ | frequency ratio, $r = \omega / \omega_n$ | dimensionless |
| $F$ | force | N |
| $x$ | displacement | m (plotted in µm) |
| $X/F$ | frequency response function (FRF) | µm/N |
| $H_1$ | measured FRF, H1 estimator | µm/N |
| $T_c$ | hammer tip contact time | s |
| $f_s$ | audio sample rate | Hz |

In the code cells, $\omega_n$ appears as `wn`, $\zeta$ as `zeta`, and $f_n$ as `fn`, matching this table; suffixed copies such as `fn_r` or `wn_x` are local to a single cell and carry the same meanings and units.


## Setup: Install and Import Libraries

Run the install cell once. If this is the first install on your machine, restart the kernel afterward (Kernel menu, Restart), then continue from the import cell.


In [ ]:
%pip install -q numpy matplotlib plotly


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, Audio, display
import plotly.graph_objects as go
from plotly.subplots import make_subplots

rcParams.update({
    'font.size': 14,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'figure.figsize': (8, 5),
    'lines.linewidth': 1.5,
    'figure.dpi': 100,
})
plt.rcParams['animation.embed_limit'] = 40  # MB

AUDIO_RATE = 48000  # Hz, sample rate for all synthesized sound

print('Libraries loaded.')


The utility functions used throughout the lesson are defined in the collapsed cell below and documented in the Appendix.


In [ ]:
# --- Utility functions used throughout the lesson (documented in the Appendix) ---
def fade(x, ms=15, fs=AUDIO_RATE):
    """Apply a short raised-cosine fade-in and fade-out so audio clips start
    and stop without clicks."""
    n = int(ms * 1e-3 * fs)
    ramp = 0.5 * (1 - np.cos(np.pi * np.arange(n) / n))
    y = x.copy()
    y[:n] *= ramp
    y[-n:] *= ramp[::-1]
    return y

def compute_spectrum(x, fs, fmax=4000):
    """Amplitude spectrum of signal x on a linear scale, normalized to a peak
    value of one. Returns the frequency vector (Hz, limited to fmax) and the
    normalized spectrum."""
    x = np.asarray(x, dtype=float)
    w = np.hanning(len(x))
    X = np.fft.rfft((x - x.mean()) * w)
    f = np.fft.rfftfreq(len(x), 1 / fs)
    amp = 2 * np.abs(X) / np.sum(w)
    amp = amp / amp.max()
    sel = f <= fmax
    return f[sel], amp[sel]

def compute_spectrogram(x, fs, nfft=2048, hop=512, fmax=4000):
    """Short-time amplitude spectrogram on a linear scale, normalized to a
    peak value of one. Returns (t_frames, f, S) where S has one column per
    time frame and one row per frequency."""
    w = np.hanning(nfft)
    n_frames = 1 + max(0, (len(x) - nfft) // hop)
    f = np.fft.rfftfreq(nfft, 1 / fs)
    sel = f <= fmax
    S = np.empty((int(sel.sum()), n_frames))
    for j in range(n_frames):
        S[:, j] = np.abs(np.fft.rfft(x[j * hop:j * hop + nfft] * w))[sel]
    S /= S.max()
    t_frames = (np.arange(n_frames) * hop + nfft / 2) / fs
    return t_frames, f[sel], S

def decimate_for_plot(t, x, max_points=None):
    """Pass plotting data through unchanged by default (max_points=None).
    To thin a long record for display, set max_points to an integer: each
    output bin then keeps its local minimum and maximum, preserving the
    visual envelope without aliasing. Computations always use the
    full-rate data."""
    n = len(x)
    if max_points is None or n <= max_points:
        return np.asarray(t), np.asarray(x)
    bins = max_points // 2
    edge = (n // bins) * bins
    xb = np.asarray(x)[:edge].reshape(bins, -1)
    tb = np.asarray(t)[:edge].reshape(bins, -1)
    tt = np.repeat(tb.mean(axis=1), 2)
    xx = np.empty(2 * bins)
    xx[0::2] = xb.min(axis=1)
    xx[1::2] = xb.max(axis=1)
    return tt, xx

def half_sine_pulse(Tc, Fmax=200.0, dur=1.0, fs=AUDIO_RATE):
    """Half-sine hammer impact of contact time Tc (s) and peak force Fmax (N)
    in a record of length dur (s). Returns the time vector and force record."""
    t = np.arange(int(dur * fs)) / fs
    F = np.where(t <= Tc, Fmax * np.sin(np.pi * t / Tc), 0.0)
    return t, F

def sdof_response(F, fn=900.0, zeta=0.02, k=8e6, fs=AUDIO_RATE):
    """Displacement response (m) of the Sect. A.3 tool model to a force
    record F (N), by semi-implicit Euler integration of Eq. 2.24 with the
    force applied on the right side."""
    wn = 2 * np.pi * fn
    m = k / wn**2
    c = 2 * zeta * np.sqrt(m * k)
    dt = 1 / fs
    x = np.zeros(len(F))
    v = 0.0
    for i in range(1, len(F)):
        v += dt * (F[i - 1] - c * v - k * x[i - 1]) / m
        x[i] = x[i - 1] + dt * v
    return x


---

## A.1 A Short History of Acoustics

Acoustics developed early because its central objects, strings and pipes, were available for careful experiment. Galileo connected pitch to the frequency of vibration in the *Two New Sciences* of 1638 [3]. Mersenne published the quantitative laws of the vibrating string, relating frequency to length, tension, and mass, at nearly the same time [4]. Boyle showed in 1660 that sound requires a medium, by pumping the air from a vessel containing a ticking watch until the ticking could no longer be heard [5].

Newton calculated the speed of sound in air in the *Principia* of 1687, obtaining a value about 15 percent below measurement [6]; Laplace supplied the correction in 1816, accounting for the heating of the air during the rapid compressions of a sound wave. In 1822 Fourier stated the theorem this lesson series depends on, that a periodic function can be written as a sum of sinusoids [7]. Helmholtz explained timbre as the ear's perception of the strengths of those harmonics in 1863 [2], and Rayleigh's *The Theory of Sound* of 1877 organized the mechanics of vibrating systems and the propagation of sound into the mathematical treatment still in use [1]. The twentieth century added the sampling theory of Nyquist and Shannon and the fast Fourier transform of Cooley and Tukey [8], the basis of the computations in this notebook.

Chatter entered the machining literature with F. W. Taylor's 1907 paper *On the Art of Cutting Metals*, which called it "the most obscure and delicate of all problems facing the machinist" [9]. Its mechanism, regeneration, was identified by Tobias, Tlusty, and their contemporaries in the 1950s and 1960s, and is the subject of Part C.


---

## A.2 The Three Kinds of Vibration

All vibrating systems can be described by one of three categories [10]:

1. *Free vibration* occurs when a system is displaced from its equilibrium position, released, and allowed to vibrate on its own. It vibrates at its natural frequency, and, in any real system, the motion decays away. A struck tuning fork is free vibration. So is a tapped end mill.

2. *Forced vibration* occurs when a system is driven by an external force that continues in time. The system vibrates at the *forcing* frequency, whatever that happens to be, with an amplitude that depends on how close the forcing frequency is to the natural frequency. An unbalanced washing machine on spin cycle is forced vibration. So is a milling cut, where the teeth strike the workpiece in a steady rhythm.

3. *Self-excited vibration* occurs when a steady input of energy is converted, by the system itself, into vibration at or near its own natural frequency. No external vibrating force is required; the system generates its own. A bowed violin string is self-excited vibration. So is chatter.

This classification is the backbone of the whole lesson. Each kind of vibration produces a characteristic sound, and learning to tell them apart by ear is the practical skill we are building. Listen to the three previews below: the ring of a tapped tool (free), the hum of a stable cut (forced), and the harsh mixture of a chattering cut (self-excited, with its mechanism deferred to Parts B and C).

▸ **Key terms: free vibration, forced vibration, self-excited vibration**


In [ ]:
# --- The three kinds of vibration, as sound ---
dur = 2.0
t = np.arange(int(dur * AUDIO_RATE)) / AUDIO_RATE

# free vibration: ring-down of a tool-like system (fn = 900 Hz, zeta = 0.02)
fn, zeta = 900.0, 0.02
wn = 2 * np.pi * fn
wd = wn * np.sqrt(1 - zeta**2)
free = 0.8 * np.exp(-zeta * wn * t) * np.sin(wd * t)

# forced vibration: steady hum at a 250 Hz forcing frequency and its harmonics
forced = np.zeros_like(t)
for k in range(1, 9):
    forced += (1 / k) * np.sin(2 * np.pi * k * 250 * t)
forced *= 0.5 / np.max(np.abs(forced))

# self-excited vibration preview: the same hum plus a strong tone near the
# natural frequency that does not belong to the 250 Hz family
self_excited = forced + 0.7 * np.sin(2 * np.pi * 913 * t)
self_excited *= 0.5 / np.max(np.abs(self_excited))

fig = make_subplots(rows=1, cols=3, shared_yaxes=True,
                    subplot_titles=('free (ring-down)', 'forced (steady hum)',
                                    'self-excited (harsh mixture)'))
n_show = int(0.04 * AUDIO_RATE)
for col, sig in enumerate([free, forced, self_excited], start=1):
    tt, xx = decimate_for_plot(t[:n_show] * 1e3, sig[:n_show])
    fig.add_trace(go.Scatter(x=tt, y=xx, mode='lines',
                             line=dict(color='#1f77b4', width=1),
                             showlegend=False), row=1, col=col)
    fig.update_xaxes(title_text='t (ms)', row=1, col=col)
fig.update_yaxes(title_text='amplitude', row=1, col=1)
fig.update_layout(title='Fig. A.1 — The three kinds of vibration (first 40 ms of each)',
                  template='simple_white', height=340,
                  margin=dict(l=60, r=20, t=80, b=50))
fig.show()

for name, sig in [('Free vibration (a tapped tool)', free),
                  ('Forced vibration (a stable cut)', forced),
                  ('Self-excited vibration (chatter, preview only)', self_excited)]:
    print(name)
    display(Audio(fade(sig), rate=AUDIO_RATE))


<div style="background-color: #e8f4fd; border-left: 5px solid #2196F3; padding: 12px 16px; margin: 12px 0; border-radius: 4px;">
<strong>Let's Talk About: Telling the Three Apart by Ear</strong><br>
The free vibration dies away on its own; if the sound persists, something is still supplying energy. The forced vibration is steady and clean, and its pitch is set entirely by the forcing, not by the machine; change the forcing rhythm and the pitch follows. The self-excited vibration is the troublemaker: the system converts a steady energy supply into vibration at a frequency of its own choosing, near its natural frequency, so the sound contains a component that does not follow the forcing rhythm. In the third clip above, the 250 Hz hum and its multiples are the forced part, and the 913 Hz tone riding on top belongs to no family of the forcing. In later Parts we detect chatter by finding such a tone.
</div>


---

## A.3 The Lumped Parameter Model

We begin with a simple, lumped parameter model where all the mass is concentrated at a single coordinate location and the spring is massless. A mass $m$ is attached to a linear spring $k$ that provides a restoring force proportional to displacement from equilibrium. For free vibration with no damping, the equation of motion is:

$$m\ddot{x} + kx = 0 \tag{2.1}$$

Two physical ingredients appear here, and only two: an elastic restoring force, described by the stiffness $k$, and inertia, described by the mass $m$. Displace the mass, and the spring pulls it back; inertia carries it through equilibrium and it overshoots; the spring pulls it back again. The result is oscillation at the *natural frequency*:

$$\omega_n = \sqrt{\frac{k}{m}}$$

in rad/s, or $f_n = \omega_n / (2\pi)$ in Hz. Stiffer systems vibrate faster; heavier systems vibrate slower. The motion itself is a cosine at that frequency:

$$x = 2A\cos(\omega_n t + \beta) \tag{2.14}$$

where the amplitude $A$ and phase $\beta$ come from how the motion was started (the initial displacement and velocity). Without damping this vibration would continue forever, which no real system does; damping is the subject of the next section.

A milling tool clamped in its holder is, to a good first approximation, exactly this system. Typical tool-point values are a stiffness of a few to a few tens of N/µm and natural frequencies from several hundred Hz to several kHz, squarely inside the range of human hearing. Let's compute one.

▸ **Key terms: stiffness, mass, natural frequency**


In [ ]:
# --- A tool-like example: from k and fn to m ---
k = 8e6            # N/m (8 N/um), a representative tool-point stiffness
fn = 900.0         # Hz, a representative tool-point natural frequency
wn = 2 * np.pi * fn
m = k / wn**2      # the mass consistent with these values

print(f'Stiffness k = {k:.1e} N/m ({k / 1e6:.0f} N/um)')
print(f'Natural frequency fn = {fn:.0f} Hz (omega_n = {wn:.0f} rad/s)')
print(f'Equivalent mass m = k / omega_n^2 = {m:.3f} kg')


#### Animation: Spring-Mass Free Vibration

The animation pairs the physical system (the mass on its spring) with the trace its motion draws over time. The red dot marks the current position on the trace. This pairing, the object on one side and its signal on the other, is how we will look at every system in this lesson.


In [ ]:
# --- Animation: undamped spring-mass drawing its displacement trace ---
f_anim = 2.0   # Hz. Try changing this!
A_anim = 1.0

t_anim = np.linspace(0, 2.0, 200)
x_anim = A_anim * np.sin(2 * np.pi * f_anim * t_anim)

fig_a, (ax_sys, ax_tr) = plt.subplots(1, 2, figsize=(12, 5),
                                      gridspec_kw={'width_ratios': [1, 2]})
plt.close(fig_a)  # Prevent static display of empty figure

def spring_pts(y_top, y_bot, n_coils=8, width=0.22):
    n_pts = 2 * n_coils + 2
    y_pts = np.linspace(y_top, y_bot, n_pts)
    x_pts = np.zeros(n_pts)
    x_pts[1:-1] = width * (-1) ** np.arange(2 * n_coils)
    return x_pts, y_pts

def init():
    return []

def animate(frame):
    ax_sys.cla()
    ax_sys.set_xlim(-1.5, 1.5)
    ax_sys.set_ylim(-2.6, 2.6)
    ax_sys.set_aspect('equal')
    ax_sys.set_axis_off()
    ax_sys.set_title('Spring-mass system (Eq. 2.1)')
    ax_sys.plot([-0.8, 0.8], [2.2, 2.2], 'k-', lw=3)          # fixed wall
    mass_y = x_anim[frame]
    sx, sy = spring_pts(2.2, mass_y + 0.25)
    ax_sys.plot(sx, sy, 'k-', lw=1.5)
    rect = plt.Rectangle((-0.4, mass_y - 0.25), 0.8, 0.5,
                         fc='#1f77b4', ec='black', lw=2)
    ax_sys.add_patch(rect)

    ax_tr.cla()
    ax_tr.set_xlim(0, t_anim[-1])
    ax_tr.set_ylim(-1.3, 1.3)
    ax_tr.set_xlabel('t (s)')
    ax_tr.set_ylabel('x')
    ax_tr.set_title(f'x(t) at f = {f_anim} Hz (Eq. 2.14)')
    ax_tr.grid(True, alpha=0.3)
    ax_tr.plot(t_anim[:frame + 1], x_anim[:frame + 1], color='#1f77b4')
    ax_tr.plot(t_anim[frame], x_anim[frame], 'o', color='#d62728', markersize=6)
    return []

anim = FuncAnimation(fig_a, animate, init_func=init,
                     frames=len(t_anim), interval=40, blit=False)
HTML(anim.to_jshtml())


<div style="background-color: #fff3e0; border-left: 5px solid #FF9800; padding: 12px 16px; margin: 12px 0; border-radius: 4px;">
<strong>In Practice</strong><br>
Hold a steel ruler flat on a desk with part of it hanging over the edge, pluck the free end, and listen. Now shorten the overhang and pluck again: the pitch rises. Shortening the overhang stiffens the ruler (larger k) and reduces the vibrating mass (smaller m), and both changes raise the natural frequency. The same reasoning applies at the machine: a long tool overhang means a lower natural frequency and a more flexible tool point, which is why tool stick-out matters so much in practice.
</div>


---

## A.4 Damping

The undamped model vibrates forever, but every real vibration decays. The mechanisms that remove energy from a vibrating system are collectively called damping, and Chap. 2 of Schmitz and Smith [10] describes three. *Viscous damping* produces a force proportional to velocity:

$$f = c\dot{x} \tag{2.15}$$

where $c$ is the viscous damping coefficient. *Coulomb damping* is the constant friction force of dry sliding surfaces, and *solid damping* (also called structural or hysteretic damping) arises from energy dissipated internally within the material itself. Real assemblies exhibit a combination, but for the remainder of this lesson we use viscous damping, which captures the observed behavior of tool-holder-spindle assemblies well and keeps the mathematics linear. The free vibration equation of motion becomes:

$$m\ddot{x} + c\dot{x} + kx = 0 \tag{2.24}$$

The character of the motion depends on how large the damping is. For the underdamped case, the typical case for machining tools, we define the *damping ratio* and the *damped natural frequency*:

$$\zeta = \frac{c}{2\sqrt{km}}, \qquad \omega_d = \omega_n\sqrt{1 - \zeta^2}$$

and the motion is an oscillation at $\omega_d$ inside an exponentially decaying envelope:

$$x = e^{-\zeta\omega_n t}\left(X_1 e^{i\omega_d t} + X_2 e^{-i\omega_d t}\right) \tag{2.29}$$

For tool points, $\zeta$ is small, typically a few percent, so $\omega_d$ is nearly equal to $\omega_n$ and the practical effect of damping is the decay rate: larger $\zeta$, faster decay. The explorer below shows the decay for a range of damping ratios; the response is immediate because every curve is precomputed.

▸ **Key terms: damping, viscous damping, damping ratio, damped natural frequency**


In [ ]:
# --- Interactive: ring-down shape versus damping ratio (client-side slider) ---
fn_r = 900.0
wn_r = 2 * np.pi * fn_r
t_r = np.linspace(0, 0.25, 900)
zetas = [0.01, 0.015, 0.02, 0.03, 0.05, 0.08, 0.12, 0.20]
START = 3   # slider starts at zeta = 0.03

fig = go.Figure()
for i, z in enumerate(zetas):
    wd_r = wn_r * np.sqrt(1 - z**2)
    env = np.exp(-z * wn_r * t_r)
    x_r = env * np.sin(wd_r * t_r)
    vis = (i == START)
    fig.add_trace(go.Scatter(x=t_r, y=x_r, mode='lines',
                             line=dict(color='#1f77b4'), name='x(t)', visible=vis))
    fig.add_trace(go.Scatter(x=t_r, y=env, mode='lines',
                             line=dict(color='#d62728', dash='dash'), opacity=0.5,
                             name='envelope', visible=vis))
    fig.add_trace(go.Scatter(x=t_r, y=-env, mode='lines',
                             line=dict(color='#d62728', dash='dash'), opacity=0.5,
                             showlegend=False, visible=vis))

steps = []
for i, z in enumerate(zetas):
    vis = [False] * (3 * len(zetas))
    vis[3 * i:3 * i + 3] = [True, True, True]
    steps.append(dict(method='update', label=f'{z:.3f}',
                      args=[{'visible': vis}]))

fig.update_layout(
    sliders=[dict(active=START, currentvalue=dict(prefix='zeta = '), steps=steps)],
    title='Free vibration of Eq. 2.24 (drag the slider)',
    xaxis_title='t (s)', yaxis_title='x (normalized)',
    yaxis_range=[-1.05, 1.05], template='simple_white', height=430,
    margin=dict(l=60, r=20, t=60, b=90))
fig.show()


#### Animation: Spring-Mass-Damper Free Vibration

The animation below shows a spring-mass-damper system vibrating freely. The mass oscillates at the damped natural frequency while the exponential envelope decays. Adjust the damping ratio in the code to see how the decay rate changes.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

plt.rcParams['animation.embed_limit'] = 40  # MB

# --- Animation: Spring-mass free vibration ---
# Parameters
m_anim = 1.0
k_anim = 1e6
zeta_anim = 0.03  # Try changing this!
wn_anim = np.sqrt(k_anim / m_anim)
wd_anim = wn_anim * np.sqrt(1 - zeta_anim**2)
x0_anim = 1.0  # mm
t_anim = np.linspace(0, 0.06, 300)
x_anim = x0_anim * np.exp(-zeta_anim * wn_anim * t_anim) * np.cos(wd_anim * t_anim)
env_anim = x0_anim * np.exp(-zeta_anim * wn_anim * t_anim)

fig_a, (ax_spring, ax_trace) = plt.subplots(1, 2, figsize=(12, 5),
                                              gridspec_kw={'width_ratios': [1, 2]})
plt.close(fig_a)

# Spring drawing helper
def draw_spring(ax, y_top, y_bot, x_center=0, n_coils=8, width=0.3):
    """Draw a zigzag spring between y_top and y_bot."""
    n_pts = 4 * n_coils + 2
    y_pts = np.linspace(y_top, y_bot, n_pts)
    x_pts = np.zeros(n_pts)
    for i in range(1, n_pts - 1):
        if i % 2 == 1:
            x_pts[i] = x_center + width * ((-1)**(i//2))
        else:
            x_pts[i] = x_center
    x_pts[0] = x_center
    x_pts[-1] = x_center
    return x_pts, y_pts

# Damper drawing helper
def draw_damper(ax, y_top, y_bot, x_center=0.5, width=0.15):
    """Draw a simple damper: rod with two parallel lines at the midpoint."""
    y_mid = (y_top + y_bot) / 2
    gap = 0.08

    # Rod from wall to mass
    ax.plot([x_center, x_center], [y_top, y_bot], 'k-', lw=1.5)

    # Two parallel horizontal lines at midpoint
    ax.plot([x_center - width, x_center + width],
            [y_mid - gap, y_mid - gap], 'k-', lw=2)
    ax.plot([x_center - width, x_center + width],
            [y_mid + gap, y_mid + gap], 'k-', lw=2)

def init():
    return []

def animate(frame):
    ax_spring.cla()
    ax_spring.set_xlim(-1.5, 1.5)
    ax_spring.set_ylim(-2.5, 2.5)
    ax_spring.set_aspect('equal')
    ax_spring.set_axis_off()
    ax_spring.set_title('Spring-Mass-Damper System')

    disp = x_anim[frame]
    mass_y = disp * 1.5  # scale for visibility

    # Fixed wall at top
    ax_spring.plot([-0.8, 0.8], [2.0, 2.0], 'k-', linewidth=3)
    ax_spring.fill_between([-0.8, 0.8], 2.0, 2.2, color='gray', alpha=0.5)

    # Mass dimensions
    mass_w = 0.8
    mass_h = 0.5
    mass_top = mass_y + mass_h / 2

    # Spring (offset to the left)
    sx, sy = draw_spring(ax_spring, 2.0, mass_top, x_center=-0.25, n_coils=8, width=0.2)
    ax_spring.plot(sx, sy, 'k-', linewidth=1.5)

    # Damper (offset to the right)
    draw_damper(ax_spring, 2.0, mass_top, x_center=0.35, width=0.15)

    # Mass (rectangle)
    rect = plt.Rectangle((-mass_w/2, mass_y - mass_h/2), mass_w, mass_h,
                          fc='#d62728', ec='black', linewidth=2)
    ax_spring.add_patch(rect)

    # Trace
    ax_trace.cla()
    ax_trace.set_xlim(0, t_anim[-1]*1e3)
    ax_trace.set_ylim(-1.2, 1.2)
    ax_trace.set_xlabel('t (ms)')
    ax_trace.set_ylabel('x (mm)')
    ax_trace.set_title(f'Displacement ($\\zeta$ = {zeta_anim})')
    ax_trace.grid(True, alpha=0.3)
    ax_trace.plot(t_anim[:frame+1]*1e3, x_anim[:frame+1], color='#1f77b4', linewidth=1.5)
    ax_trace.plot(t_anim*1e3, env_anim, '--', color='#d62728', alpha=0.3)
    ax_trace.plot(t_anim*1e3, -env_anim, '--', color='#d62728', alpha=0.3)
    ax_trace.plot(t_anim[frame]*1e3, x_anim[frame], 'o', color='#d62728', markersize=6)

    return []

anim = FuncAnimation(fig_a, animate, init_func=init,
                     frames=len(t_anim), interval=40, blit=False)
HTML(anim.to_jshtml())


<div style="background-color: #fff3e0; border-left: 5px solid #FF9800; padding: 12px 16px; margin: 12px 0; border-radius: 4px;">
<strong>In Practice</strong><br>
Tap an empty wine glass and it rings for seconds: the glass dissipates very little energy per cycle, so its damping ratio is small. Now fill that same glass with water and tap it again: the ring dies almost immediately, because the water sloshing against the wall carries energy away with every cycle. Nothing about the glass changed except how much dissipation was attached to it. Machine tools acquire their damping the same way, from wherever vibration energy can be dissipated: friction and micro-slip at bolted and clamped joints, the tool-holder and holder-spindle interfaces, bearings and guideways, and the internal (solid) damping of the structural materials themselves. Because these sources are difficult to predict from first principles, damping is the parameter we measure rather than calculate.
</div>


---

## A.5 Forced Vibration and Resonance

Now we add an external harmonic force to the spring-mass-damper. The equation of motion becomes:

$$m\ddot{x} + c\dot{x} + kx = fe^{i\omega t} \tag{2.31}$$

where $\omega$ is the forcing frequency, chosen by whatever supplies the force, not by the system. After any initial transients decay, the system settles into the steady-state response: it vibrates at the forcing frequency $\omega$, with an amplitude and a time delay (phase) that depend on where $\omega$ sits relative to the natural frequency. It is convenient to describe that relationship using the frequency ratio $r = \omega / \omega_n$:

- Well below the natural frequency ($r \ll 1$), the response is quasi-static: the mass simply follows the force, with amplitude set by the stiffness alone.
- Near the natural frequency ($r = 1$), the response amplitude grows dramatically. This is *resonance*, and for light damping the amplification is large: the amplitude at resonance is a factor of $1/(2\zeta)$ times the static deflection.
- Well above the natural frequency ($r \gg 1$), the mass cannot keep up with the rapidly reversing force and the response shrinks toward zero.

The animation below sweeps the forcing frequency through resonance, and the audio clip after it lets you hear the same sweep: a tone whose loudness follows the response amplitude, swelling as it passes the natural frequency.

▸ **Key terms: forcing frequency, steady state, resonance**


#### Animation: Forced Vibration Through Resonance

The animation below sweeps the forcing frequency from well below to well above the natural frequency. Watch how the displacement magnitude grows dramatically near resonance ($r = 1$) and how the phase shifts from $0°$ to $-180°$.


In [ ]:
# --- Animation: Forced vibration frequency sweep ---
m_fv = 1.0
k_fv = 1e6
zeta_fv = 0.05
wn_fv = np.sqrt(k_fv / m_fv)

# Sweep r from 0.1 to 2.0
r_sweep = np.linspace(0.1, 2.0, 200)

# Precompute FRF magnitude and phase for each r
mag_sweep = (1/k_fv) / np.sqrt((1 - r_sweep**2)**2 + (2*zeta_fv*r_sweep)**2)
phase_sweep = np.arctan2(-2*zeta_fv*r_sweep, 1 - r_sweep**2)

# Time vector for one "snapshot" of oscillation
t_snap = np.linspace(0, 4*np.pi, 300)  # 2 full cycles of forcing

fig_fv, axes_fv = plt.subplots(1, 3, figsize=(15, 4),
                                gridspec_kw={'width_ratios': [2, 1, 1]})
plt.close(fig_fv)

def init_fv():
    return []

def animate_fv(frame):
    for ax in axes_fv:
        ax.cla()

    r_val = r_sweep[frame]
    mag_val = mag_sweep[frame]
    phase_val = phase_sweep[frame]
    w_force = r_val * wn_fv

    # Left panel: force and displacement vs time
    force_signal = np.cos(t_snap)
    disp_signal = mag_val * k_fv * np.cos(t_snap + phase_val)  # normalized

    ax0 = axes_fv[0]
    ax0.plot(t_snap/(2*np.pi), force_signal, 'k-', alpha=0.4, label='Force (normalized)')
    ax0.plot(t_snap/(2*np.pi), disp_signal, color='#1f77b4', linewidth=2, label='Displacement (normalized)')
    ax0.set_xlim([0, 2])
    ax0.set_ylim([-12, 12])
    ax0.set_xlabel('Cycles')
    ax0.set_ylabel('Amplitude')
    ax0.set_title(f'r = {r_val:.2f}, f = {w_force/(2*np.pi):.0f} Hz')
    ax0.legend(loc='upper right', fontsize=9)
    ax0.grid(True, alpha=0.3)

    # Middle panel: magnitude on FRF curve
    r_full = np.linspace(0, 2, 500)
    mag_full = (1/k_fv) / np.sqrt((1 - r_full**2)**2 + (2*zeta_fv*r_full)**2)

    ax1 = axes_fv[1]
    ax1.plot(r_full, mag_full * 1e6, color='#1f77b4', alpha=0.4)
    ax1.plot(r_val, mag_val * 1e6, 'o', color='#d62728', markersize=8)
    ax1.set_xlabel('r')
    ax1.set_ylabel('|X/F| ($\\times 10^{-6}$ m/N)')
    ax1.set_title('Magnitude')
    ax1.set_xlim([0, 2])
    ax1.set_ylim([0, 12])
    ax1.grid(True, alpha=0.3)

    # Right panel: phase on FRF curve
    phase_full = np.degrees(np.arctan2(-2*zeta_fv*r_full, 1 - r_full**2))

    ax2 = axes_fv[2]
    ax2.plot(r_full, phase_full, color='#1f77b4', alpha=0.4)
    ax2.plot(r_val, np.degrees(phase_val), 'o', color='#d62728', markersize=8)
    ax2.set_xlabel('r')
    ax2.set_ylabel('Phase (deg)')
    ax2.set_title('Phase')
    ax2.set_xlim([0, 2])
    ax2.set_ylim([-185, 5])
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    return []

anim_fv = FuncAnimation(fig_fv, animate_fv, init_func=init_fv,
                        frames=len(r_sweep), interval=60, blit=False)
HTML(anim_fv.to_jshtml())


In [ ]:
# --- Hearing resonance: a force sweep through the natural frequency ---
fn_s = 900.0        # Hz, natural frequency of the system being driven
zeta_s = 0.03
dur = 5.0
t_s = np.arange(int(dur * AUDIO_RATE)) / AUDIO_RATE

# forcing frequency sweeps linearly from 300 Hz to 1800 Hz
f_of_t = 300 + (1800 - 300) * t_s / dur
phase = 2 * np.pi * np.cumsum(f_of_t) / AUDIO_RATE

# response amplitude follows the FRF magnitude at the instantaneous r
r_of_t = f_of_t / fn_s
amp = 1 / np.sqrt((1 - r_of_t**2)**2 + (2 * zeta_s * r_of_t)**2)
amp /= amp.max()

sweep_sound = fade(0.8 * amp * np.sin(phase))
t_res = dur * (fn_s - 300) / (1800 - 300)
print(f'The forcing sweeps 300 to 1800 Hz over {dur:.0f} s at constant force amplitude.')
print(f'Listen for the swell as it crosses the natural frequency of {fn_s:.0f} Hz,')
print(f'about {t_res:.1f} s in. The force amplitude is constant throughout; the rise')
print('in loudness is amplification by the system.')
display(Audio(sweep_sound, rate=AUDIO_RATE))


<div style="background-color: #e8f4fd; border-left: 5px solid #2196F3; padding: 12px 16px; margin: 12px 0; border-radius: 4px;">
<strong>Let's Talk About: Pushing a Swing</strong><br>
Resonance is familiar from the playground. A swing has a natural frequency set by its length, and a sequence of small, well-timed pushes builds a large motion because each push arrives in step with the motion. Push at the wrong rhythm and the same effort accomplishes little. The machine-shop translation: a milling cut applies its force in a steady rhythm set by the spindle, and the tool has a natural frequency of its own. How those two frequencies line up governs the vibration amplitude, and, as we will see in Parts B and C, it also governs stability. Selecting the spindle speed sets the timing of those pushes.
</div>


---

## A.6 The Frequency Response Function

Sect. A.5 described the steady-state response at one forcing frequency. Collecting that result for every forcing frequency gives the *frequency response function (FRF)*, the single most important quantity in this lesson [10]. From the steady-state solution of Eq. 2.31:

$$\frac{X}{F} = \frac{1}{k - m\omega^2 + ic\omega} \tag{2.33}$$

or, dividing by $k$ and introducing the frequency ratio $r = \omega / \omega_n$:

$$\frac{X}{F} = \frac{1}{k} \cdot \frac{1}{(1-r^2) + i(2\zeta r)} \tag{2.34}$$

The FRF is complex valued: at each frequency it stores both the amplitude ratio (how many µm of vibration per N of force) and the phase (the time delay between force and response). It is often displayed as its real and imaginary parts:

$$\text{Re}\left(\frac{X}{F}\right) = \frac{1}{k} \cdot \frac{1-r^2}{(1-r^2)^2 + (2\zeta r)^2} \tag{2.36}$$

$$\text{Im}\left(\frac{X}{F}\right) = \frac{1}{k} \cdot \frac{-2\zeta r}{(1-r^2)^2 + (2\zeta r)^2} \tag{2.37}$$

Three features anchor the picture. At $r = 0$ the magnitude is $1/k$, the static compliance. At $r = 1$ the magnitude peaks at $1/(2k\zeta)$ and the phase passes through $-90°$. Well above resonance the response falls toward zero. The FRF is the fingerprint of a structure: every tool, holder, and spindle combination has its own, and it describes how the assembly responds to any steady forcing.

Equation 2.34 also tells us how to explore the FRF efficiently: the product $k \cdot X/F$ depends only on $r$ and $\zeta$. The natural frequency places the resonance on the frequency axis and the stiffness sets the µm/N scale, but the shape of the curve is governed by the damping ratio alone. The explorer below plots the normalized FRF against $r$; drag the slider and $\zeta$ reshapes all four views at once, instantly, because every curve is precomputed.

▸ **Key terms: frequency response function (FRF), static compliance**


In [ ]:
# --- Interactive: the normalized SDOF FRF shaped by zeta (client-side slider) ---
r = np.linspace(0.01, 2.5, 700)
zetas_f = [0.01, 0.015, 0.02, 0.03, 0.05, 0.08, 0.12, 0.20]
START_F = 3   # slider starts at zeta = 0.03

fig = make_subplots(rows=2, cols=2, shared_xaxes=True,
                    subplot_titles=('Magnitude k|X/F|', 'Phase (deg)',
                                    'Real part (Eq. 2.36)', 'Imaginary part (Eq. 2.37)'))
for i, z in enumerate(zetas_f):
    H = 1.0 / ((1 - r**2) + 1j * 2 * z * r)   # k * X/F, Eq. 2.34
    vis = (i == START_F)
    line = dict(color='#1f77b4')
    fig.add_trace(go.Scatter(x=r, y=np.abs(H), mode='lines', line=line,
                             visible=vis, showlegend=False), row=1, col=1)
    fig.add_trace(go.Scatter(x=r, y=np.degrees(np.angle(H)), mode='lines', line=line,
                             visible=vis, showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=r, y=H.real, mode='lines', line=line,
                             visible=vis, showlegend=False), row=2, col=1)
    fig.add_trace(go.Scatter(x=r, y=H.imag, mode='lines', line=line,
                             visible=vis, showlegend=False), row=2, col=2)

steps = []
for i, z in enumerate(zetas_f):
    vis = [False] * (4 * len(zetas_f))
    vis[4 * i:4 * i + 4] = [True] * 4
    steps.append(dict(method='update', label=f'{z:.3f}', args=[{'visible': vis}]))

fig.update_layout(
    sliders=[dict(active=START_F, currentvalue=dict(prefix='zeta = '), steps=steps)],
    template='simple_white', height=560, showlegend=False,
    margin=dict(l=60, r=20, t=60, b=90))
fig.update_xaxes(title_text='frequency ratio r', row=2, col=1)
fig.update_xaxes(title_text='frequency ratio r', row=2, col=2)
fig.update_yaxes(range=[0, 52], row=1, col=1)
fig.update_yaxes(range=[-185, 5], row=1, col=2)
fig.update_yaxes(range=[-27, 27], row=2, col=1)
fig.update_yaxes(range=[-52, 2], row=2, col=2)
fig.show()


In [ ]:
# --- From the normalized picture to physical units ---
fn_e = 900.0     # Hz. Try changing these!
k_e = 8.0        # N/um
zeta_e = 0.03

print(f'For fn = {fn_e:.0f} Hz, k = {k_e:.0f} N/um, zeta = {zeta_e:.3f}:')
print(f'  static compliance 1/k                    = {1 / k_e:.3f} um/N (the r = 0 value)')
print(f'  peak magnitude at resonance 1/(2 k zeta) = {1 / (2 * k_e * zeta_e):.2f} um/N '
      f'({1 / (2 * zeta_e):.0f} times the static value)')
print(f'  on a measured FRF, that peak sits at about {fn_e:.0f} Hz')


---

## A.7 The Tap Test

Everything above assumed we know $f_n$, $k$, and $\zeta$. For a real tool in a real spindle we do not; we measure them. The standard measurement is *impact testing*, better known as the tap test [10]: strike the tool point with an instrumented hammer, which applies a short, known force, and record the response with a sensor. The ratio of measured response to measured force, frequency by frequency, is the FRF of Sect. A.6.

The hammer impact is modeled well as a half-sine force pulse: the force rises from zero to its peak and returns to zero over the *contact time* $T_c$. The contact time is set by the hammer tip. A hard steel tip rebounds quickly, giving a short pulse; a soft rubber tip stays in contact longer, giving a wide one. The width matters because it sets the *excitation bandwidth*: a short pulse contains energy over a wide frequency range, while a long pulse concentrates its energy at low frequencies. As a working rule, a half-sine pulse of contact time $T_c$ excites frequencies up to roughly $1/T_c$, and its energy falls off steeply beyond that. The tip selection sets the bandwidth of the test, and the tip must excite the frequencies we intend to measure.

In this section we simulate the entire measurement: the half-sine impact for three tips, the free vibration response of the Sect. A.3 tool model to each, the sound of each tap, and finally the FRF computed as the ratio of the response spectrum to the force spectrum, compared against the exact result of Eq. 2.33.

The response sensor is commonly a small accelerometer waxed to the tool, but a microphone works too: the ringing tool radiates its vibration into the air, and the recorded sound contains the same frequencies. A microphone cannot deliver the calibrated force-to-response ratio needed for a full FRF, but it identifies the natural frequency directly and requires no instrumentation beyond a phone. That is the simplified tap test used later in this series; a hammer-based impact test is the full version of the same idea.

▸ **Key terms: tap test, impact testing, contact time, excitation bandwidth, H1 estimator**


In [ ]:
# --- Fig. A.2: hammer tips as half-sine pulses, and the bandwidth of each ---
TIPS = [('hard (steel)', 0.0002, '#1f77b4', 'solid'),
        ('medium (plastic)', 0.0008, '#ff7f0e', 'dash'),
        ('soft (rubber)', 0.0025, '#2ca02c', 'dot')]
TIPS_COLORS = {n: c for n, _, c, _ in TIPS}
TIPS_DASHES = {n: d for n, _, _, d in TIPS}

fig = make_subplots(rows=1, cols=2, subplot_titles=('Force pulse', 'Force spectrum'))
n_show = int(0.004 * AUDIO_RATE)
for name, Tc, color, dash in TIPS:
    t_p, F_p = half_sine_pulse(Tc)
    fig.add_trace(go.Scatter(x=t_p[:n_show] * 1e3, y=F_p[:n_show], mode='lines',
                             line=dict(color=color, dash=dash), name=name), row=1, col=1)
    Xf = np.abs(np.fft.rfft(F_p))
    Xf = Xf / Xf.max()
    f_ax = np.fft.rfftfreq(len(F_p), 1 / AUDIO_RATE)
    sel = f_ax <= 3000
    ff, aa = decimate_for_plot(f_ax[sel], Xf[sel])
    fig.add_trace(go.Scatter(x=ff, y=aa, mode='lines',
                             line=dict(color=color, dash=dash), showlegend=False), row=1, col=2)
fig.add_shape(type='line', x0=900, x1=900, y0=0, y1=1.02, xref='x2', yref='y2',
              line=dict(color='black', width=1, dash='dot'), opacity=0.5)
fig.add_annotation(x=940, y=0.92, xref='x2', yref='y2', text='900 Hz mode',
                   showarrow=False, font=dict(size=10), xanchor='left')
fig.update_xaxes(title_text='t (ms)', row=1, col=1)
fig.update_xaxes(title_text='frequency (Hz)', row=1, col=2)
fig.update_yaxes(title_text='force (N)', row=1, col=1)
fig.update_yaxes(title_text='force content (normalized)', range=[0, 1.05], row=1, col=2)
fig.update_layout(title='Fig. A.2 — Three hammer tips at equal peak force',
                  template='simple_white', height=400,
                  legend=dict(x=0.44, y=0.98, xanchor='right'),
                  margin=dict(l=60, r=20, t=70, b=50))
fig.show()

for name, Tc, color, dash in TIPS:
    print(f'{name:>17}: Tc = {Tc * 1e3:.1f} ms, excites up to roughly 1/Tc = {1 / Tc:,.0f} Hz')


Figure A.2 makes a testable prediction: the soft tip has almost no force content at 900 Hz, so it should barely wake the mode. The response follows from time-domain integration of the equation of motion, Eq. 2.24 with the impact force applied. Each strike produces a burst of free vibration at the damped natural frequency, and how strongly the mode rings depends on how much force content the tip delivered at that frequency. Watch the soft tip's trace closely: during its long contact the tool simply follows the force, a quasi-static push of roughly $F/k$, and only a weak ring survives after the tip leaves. One caution when reading the amplitudes: at equal peak force the soft tip delivers a much larger total impulse (the area under its force pulse), which partly masks its bandwidth deficit. The fair comparison is the ring produced per unit of delivered impulse, printed below.


In [ ]:
# --- Fig. A.3: the response of the 900 Hz tool model to each tip ---
responses = {}
for name, Tc, color, dash in TIPS:
    t_p, F_p = half_sine_pulse(Tc)
    x_p = sdof_response(F_p)
    responses[name] = (t_p, F_p, x_p)

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, shared_yaxes=True,
                    subplot_titles=[name for name, _, _, _ in TIPS])
n_show = int(0.06 * AUDIO_RATE)
for row, (name, Tc, color, dash) in enumerate(TIPS, start=1):
    t_p, F_p, x_p = responses[name]
    tt, xx = decimate_for_plot(t_p[:n_show] * 1e3, x_p[:n_show] * 1e6)
    fig.add_trace(go.Scatter(x=tt, y=xx, mode='lines',
                             line=dict(color=color, dash=dash),
                             showlegend=False), row=row, col=1)
    fig.update_yaxes(title_text='x (µm)', row=row, col=1)
fig.update_xaxes(title_text='t (ms)', row=3, col=1)
fig.update_layout(title='Fig. A.3 — Tool point response to each hammer tip',
                  template='simple_white', height=560,
                  margin=dict(l=60, r=20, t=70, b=50))
fig.show()

for name, Tc, color, dash in TIPS:
    t_p, F_p, x_p = responses[name]
    ring = np.max(np.abs(x_p[int((Tc + 0.002) * AUDIO_RATE):]))   # after contact ends
    impulse = np.sum(F_p) / AUDIO_RATE                            # N*s, area under the pulse
    print(f'{name:>17}: ring after the impact {ring * 1e6:5.1f} µm from an impulse of '
          f'{impulse * 1e3:6.1f} mN·s, i.e. {ring * 1e6 / impulse:4.0f} µm per N·s')


Now the measurement itself. The tap test does not read the FRF off a chart; it computes it from the two recorded signals. In practice this is done with the $H_1$ estimator, the standard formulation in impact testing [10]:

$$H_1(f) = \frac{G_{xF}(f)}{G_{FF}(f)} = \frac{X(f)\,\overline{F(f)}}{F(f)\,\overline{F(f)}}$$

where $G_{xF}$ is the cross spectrum between response and force, $G_{FF}$ is the force autospectrum, and the overbar denotes the complex conjugate. For a single record the conjugates cancel and $H_1$ reduces exactly to the ratio $X(f)/F(f)$; its value in the laboratory comes from averaging the cross and auto spectra over several taps before dividing, which suppresses measurement noise. Our simulation is a single clean tap, so the two forms coincide.

If the model and the measurement are both sound, $H_1$ should reproduce Eq. 2.33. The cell below computes it for all three tips and overlays the exact curve. Watch the soft tip above its bandwidth: where its force spectrum passes through zero, the estimator divides one vanishing quantity by another, and any tiny imperfection is amplified into large spikes. The tip bandwidth sets the frequency range over which the measurement can be trusted.


In [ ]:
# --- Fig. A.4: the measured FRF, response spectrum over force spectrum ---
f_axis = np.fft.rfftfreq(len(responses['hard (steel)'][1]), 1 / AUDIO_RATE)
sel = (f_axis > 20) & (f_axis <= 2500)

fig = go.Figure()
for name in ['hard (steel)', 'medium (plastic)', 'soft (rubber)']:
    t_p, F_p, x_p = responses[name]
    Ff = np.fft.rfft(F_p)
    H_meas = (np.fft.rfft(x_p) * np.conj(Ff)) / (Ff * np.conj(Ff))   # H1 estimator
    ff, hh = decimate_for_plot(f_axis[sel], np.abs(H_meas[sel]) * 1e6)
    fig.add_trace(go.Scatter(x=ff, y=hh, mode='lines',
                             line=dict(color=TIPS_COLORS[name],
                                       dash=TIPS_DASHES[name]), opacity=0.85,
                             name=f'measured, {name}'))

# the exact FRF of Eq. 2.33 for the same fn, zeta, k
wn_x = 2 * np.pi * 900.0
m_x = 8e6 / wn_x**2
c_x = 2 * 0.02 * np.sqrt(m_x * 8e6)
w_x = 2 * np.pi * f_axis[sel]
H_exact = 1.0 / (8e6 - m_x * w_x**2 + 1j * c_x * w_x)
fig.add_trace(go.Scatter(x=f_axis[sel], y=np.abs(H_exact) * 1e6, mode='lines',
                         line=dict(color='black', dash='dash', width=1.4),
                         name='exact, Eq. 2.33'))

fig.update_layout(title='Fig. A.4 — Measured FRF compared with Eq. 2.33',
                  template='simple_white', height=460,
                  xaxis_title='frequency (Hz)', yaxis_title='|X/F| (µm/N)',
                  yaxis_range=[0, 4], legend=dict(x=0.98, y=0.98, xanchor='right'),
                  margin=dict(l=60, r=20, t=60, b=50))
fig.show()

t_p, F_p, x_p = responses['hard (steel)']
Ff = np.fft.rfft(F_p)
H_hard = (np.fft.rfft(x_p) * np.conj(Ff)) / (Ff * np.conj(Ff))       # H1 estimator
fn_measured = f_axis[sel][np.argmax(np.abs(H_hard[sel]))]
pk = np.abs(H_hard[sel]).max() * 1e6
print(f'Hard tip measurement: FRF peak at {fn_measured:.1f} Hz, {pk:.2f} µm/N')
print(f'Exact values:         900.0 Hz, 1/(2 k zeta) = {1e6 / (2 * 8e6 * 0.02):.2f} µm/N')


The hard and medium tips reproduce Eq. 2.33 across the plotted band, peak and all, since the 900 Hz mode lies inside both bandwidths. The soft tip agrees at low frequency and then erupts exactly where Fig. A.2 showed its force spectrum passing through zero: with almost nothing delivered and almost nothing measured at those frequencies, the ratio amplifies numerical residue into spikes. Fig. A.2's prediction is confirmed, and the practical rule follows directly: trust a tap test only within the bandwidth of the tip that performed it.


<div style="background-color: #fff3e0; border-left: 5px solid #FF9800; padding: 12px 16px; margin: 12px 0; border-radius: 4px;">
<strong>In Practice</strong><br>
A production tap test uses exactly this procedure with dedicated hardware: an instrumented hammer for the force, an accelerometer or vibrometer for the response, and the computed FRF supplying $f_n$, $k$, and $\zeta$ for stability analysis. The tip choice is a real decision on the shop floor, made with the rule from Fig. A.2: the contact time must be short enough to excite the modes of interest. The phone version costs nothing to repeat: record while tapping the tool with a hex key (a hard tip, so the bandwidth is wide), and read the ring frequency from the spectrum. If a tool change or a longer stick-out moves that frequency, the dynamics changed, and the stability analysis should be repeated.
</div>


---

## A.8 From Vibration to Sound

One step remains: connecting the vibrating structure to the sound at your ear. A vibrating surface pushes on the air in front of it as it moves forward and leaves a slight rarefaction as it moves back. These pressure disturbances travel outward through the air at about 343 m/s at room temperature, each layer of air nudging the next, until they reach your ear or a microphone. *Sound* is a traveling pressure vibration, and its frequency content is the frequency content of the vibration that launched it. Human hearing spans roughly 20 Hz to 20 kHz, and tool natural frequencies, spindle rotation rates, and tooth passing rhythms all fall inside that range.

To read that broadcast we use three connected views of the same recording. The *time domain* view is the recording itself, amplitude against time: it shows when things happen, but says nothing directly about frequency. The *frequency domain* view, the *spectrum*, shows which frequencies are present over the whole record, but says nothing about when. The *spectrogram* sits between them: many short-time spectra stacked side by side, time left to right, frequency bottom to top, brightness showing energy. It trades a little frequency precision for the ability to see when each frequency occurs. All three describe the same signal; they differ only in which question they answer.

We practice on three sounds whose pictures we can predict before looking: a pure tone, a harmonic tone built from one fundamental and its multiples (the hum of Sect. A.2), and, for fun, a musical scale played up and back down. For each, ask the three questions in order: what does it do in time, which frequencies does it contain overall, and when does each frequency occur?

▸ **Key terms: sound, time domain, frequency domain, spectrum, spectrogram**


In [ ]:
# --- Fig. A.5: three sounds in three views: time, spectrum, spectrogram ---
dur = 2.0
t = np.arange(int(dur * AUDIO_RATE)) / AUDIO_RATE

# 1. a pure tone: A4, 440 Hz
pure = 0.6 * np.sin(2 * np.pi * 440 * t)

# 2. a harmonic tone: a 250 Hz fundamental with 1/k harmonics (the hum of Sect. A.2)
harmonic = np.zeros_like(t)
for k in range(1, 9):
    harmonic += (1 / k) * np.sin(2 * np.pi * k * 250 * t)
harmonic *= 0.6 / np.max(np.abs(harmonic))

# 3. a musical scale: C major, one octave up and back down
up = [261.63, 293.66, 329.63, 349.23, 392.00, 440.00, 493.88, 523.25]
notes = up + up[-2::-1]                      # C4 up to C5, then back down to C4
dur_scale = 3.0
note_len = int(dur_scale / len(notes) * AUDIO_RATE)
t_scale = np.arange(len(notes) * note_len) / AUDIO_RATE
scale_sig = np.zeros(len(notes) * note_len)
for j, f_note in enumerate(notes):
    seg = np.arange(note_len) / AUDIO_RATE
    scale_sig[j * note_len:(j + 1) * note_len] = fade(
        0.6 * np.sin(2 * np.pi * f_note * seg), ms=10)

print('Listen to each sound, then read its row below, left to right.')
for name, sig in [('1. pure tone (440 Hz)', pure),
                  ('2. harmonic tone (250 Hz and its multiples)', harmonic),
                  ('3. C major scale, up and down', scale_sig)]:
    print(name)
    display(Audio(fade(sig), rate=AUDIO_RATE))

signals = [('pure tone', t, pure), ('harmonic tone', t, harmonic),
           ('musical scale', t_scale, scale_sig)]

fig = make_subplots(rows=3, cols=3,
                    column_titles=('time domain', 'frequency domain (spectrum)',
                                   'spectrogram'),
                    row_titles=[name for name, _, _ in signals],
                    horizontal_spacing=0.07, vertical_spacing=0.08)
for row, (name, t_s, sig) in enumerate(signals, start=1):
    tt, xx = decimate_for_plot(t_s, sig)
    fig.add_trace(go.Scatter(x=tt, y=xx, mode='lines',
                             line=dict(color='#1f77b4', width=0.7),
                             showlegend=False), row=row, col=1)
    f_sp, a_sp = compute_spectrum(sig, AUDIO_RATE, fmax=2500)
    ff, aa = decimate_for_plot(f_sp, a_sp)
    fig.add_trace(go.Scatter(x=ff, y=aa, mode='lines',
                             line=dict(color='#1f77b4'),
                             showlegend=False), row=row, col=2)
    t_g, f_g, S = compute_spectrogram(sig, AUDIO_RATE, fmax=2500)
    fig.add_trace(go.Heatmap(x=t_g, y=f_g, z=S, colorscale='Inferno',
                             showscale=False), row=row, col=3)
    fig.update_yaxes(title_text='amplitude', row=row, col=1)
    fig.update_yaxes(title_text='amplitude (norm.)', range=[0, 1.05], row=row, col=2)
    fig.update_yaxes(title_text='frequency (Hz)', range=[0, 2500], row=row, col=3)
fig.update_xaxes(title_text='t (s)', row=3, col=1)
fig.update_xaxes(title_text='frequency (Hz)', row=3, col=2)
fig.update_xaxes(title_text='t (s)', row=3, col=3)
fig.update_layout(title='Fig. A.5 — The same three sounds in three views',
                  template='simple_white', height=820,
                  margin=dict(l=70, r=40, t=80, b=50))
fig.show()


Reading Fig. A.5 row by row. The pure tone is steady in time, a single peak at 440 Hz in the spectrum, and one unbroken horizontal line in the spectrogram: all three views agree because nothing changes. The harmonic tone repeats in time with a more intricate waveform, its spectrum is a comb at 250 Hz and its multiples, and its spectrogram is a stack of parallel lines; this is the picture of the forced-vibration hum from Sect. A.2, and it is the picture a healthy milling cut will draw in Part B.

The scale is the instructive row. Its time trace is nearly featureless, and its spectrum faithfully reports all eight notes while saying nothing about their order; up, down, or shuffled, the spectrum would look the same. Only the spectrogram shows the staircase climbing to the octave and stepping back down. A melody is frequency as a function of time, and the spectrogram displays it directly. Each view answers a different question, and we will use all three when we listen to a machine in Part B.


<div style="background-color: #e8f4fd; border-left: 5px solid #2196F3; padding: 12px 16px; margin: 12px 0; border-radius: 4px;">
<strong>Let's Talk About: What the Phone Can and Cannot Measure</strong><br>
Two practical notes before we point these tools at a machine. First, phones sample sound about 48,000 times per second, which is fast enough to represent all frequencies within human hearing; the details of sampling belong to Part B. Second, phone microphones are not calibrated, so the absolute loudness values in a recording are not trustworthy, but the frequency axis is. Every conclusion we draw from machine sound in this lesson rests on the frequencies of the peaks rather than on their measured heights.
</div>


That completes Part A. We can name the three kinds of vibration, compute a natural frequency, explain why the ring decays, read an FRF, measure a tool by tapping it, and relate the three views of a recorded sound. In Part B we point this toolkit at a milling machine and build its sound vocabulary: the idle baseline, the once-per-revolution content of the rotating spindle, the once-per-tooth rhythm of the cut, and what each looks like on the spectrogram.


---

## Summary

- The study of sound and vibration developed from the string experiments of Galileo and Mersenne through Fourier, Helmholtz, and Rayleigh to twentieth century sampling theory and the fast Fourier transform; chatter entered the machining literature with Taylor's 1907 description.
- All vibration is free, forced, or self-excited, and the three correspond to the ring of a tapped tool, the hum of a stable cut, and the squeal of chatter.
- The simplest vibrating system is a mass on a spring (Eq. 2.1); its natural frequency is $\omega_n = \sqrt{k/m}$, so stiffer means faster and heavier means slower.
- Damping removes energy; for the underdamped case typical of tools, the motion oscillates at $\omega_d$ inside a decaying envelope (Eq. 2.29), and the damping ratio $\zeta$ sets the decay rate.
- Under harmonic forcing (Eq. 2.31) the steady-state response occurs at the forcing frequency, with dramatic amplification near resonance, where the amplitude reaches $1/(2\zeta)$ times the static deflection.
- The FRF (Eqs. 2.33 to 2.37) collects the response at every frequency and is the fingerprint of a structure; the tap test measures it with the H1 estimator, the ratio of response spectrum to force spectrum, and the hammer tip's contact time sets the excitation bandwidth, and a phone microphone can identify the natural frequency alone.
- A vibrating structure broadcasts its frequencies into the air as sound; the time trace, the spectrum, and the spectrogram are three views of the same recording, and each answers a different question about it.

Part B builds the analysis chain behind the spectra used here, starting from what a microphone records. The tool model of Sect. A.7 returns in Part C as the structure whose cutting stability is in question.


---

## Exercises

**1.** A tool point has stiffness $k = 12$ N/µm and natural frequency $f_n = 1400$ Hz.
**(a)** What equivalent mass is consistent with these values? Check by editing the collapsed Sect. A.3 cell.
**(b)** If a longer tool reduces the stiffness to 6 N/µm while the equivalent mass stays approximately the same, what is the new natural frequency?

**2.** In the spring-mass-damper animation, set `zeta_anim = 0.10` and rerun.
**(a)** How does the trace change compared with `zeta_anim = 0.03`?
**(b)** Using the decay of the envelope, roughly how many cycles does each case take to fall to 5 percent of the starting amplitude?

**3.** Using the ring-down explorer of Sect. A.4, count the visible oscillation cycles before the envelope becomes negligible for $\zeta = 0.01$ and again for $\zeta = 0.05$.
**(a)** How does the cycle count scale with $\zeta$?
**(b)** The 5 percent decay time is $3/(\zeta\omega_n)$. Show that the corresponding number of cycles is approximately $3/(2\pi\zeta)$, and check the result against your counts.

**4.** In the FRF explorer of Sect. A.6, set $\zeta = 0.02$.
**(a)** Read the peak of the normalized magnitude $k|X/F|$ from the plot and confirm it equals $1/(2\zeta)$. For a tool with $k = 8$ N/µm, what is the peak in µm/N?
**(b)** Double the damping ratio. What happens to the resonance peak, and what happens to the rest of the curve?

**5.** In the tap test simulation of Sect. A.7:
**(a)** Change the tool model inside `sdof_response` (for example, `fn=1400.0`) and rerun the section. Does the measured FRF peak in Fig. A.4 track the change, and which tips still excite the new mode adequately?
**(b)** Add a fourth, even softer tip with $T_c = 6$ ms to the `TIPS` list. Predict its bandwidth from the $1/T_c$ rule before running, then confirm against Figs. A.2 and A.3.
**(c)** Using Fig. A.4, explain in one sentence why the soft tip's measured FRF is trustworthy below a few hundred Hz and worthless near 900 Hz.


---

## Appendix: Utility Functions

The code cells of this lesson rely on the helper functions, defined once in the collapsed cell at the top of the notebook (they must be defined before their first use so that Restart and Run All succeeds). For reference:

- `fade(x, ms, fs)` applies a short raised-cosine fade-in and fade-out so audio clips start and stop without clicks.
- `compute_spectrum(x, fs, fmax)` returns the amplitude spectrum of a signal on a linear scale, normalized to a peak of one.
- `compute_spectrogram(x, fs, nfft, hop, fmax)` returns the short-time spectrogram on the same linear, normalized scale, used for the heatmap panels.
- `decimate_for_plot(t, x, max_points)` passes plotting data through unchanged by default; set `max_points` to an integer to thin a long record for display, keeping each bin's minimum and maximum so the visual envelope is preserved without aliasing. Computations always use the full-rate data.
- `half_sine_pulse(Tc, Fmax, dur, fs)` constructs the half-sine hammer impact of contact time $T_c$ used in Sect. A.7.
- `sdof_response(F, fn, zeta, k, fs)` integrates Eq. 2.24 with an applied force record to produce the tool point displacement response.


---

## References

**[1]** Strutt, J. W. (Lord Rayleigh) (1877). *The Theory of Sound*, Vol. 1. Macmillan. Reprinted by Dover Publications, 1945.

**[2]** Helmholtz, H. (1863). *Die Lehre von den Tonempfindungen*. English translation by A. J. Ellis from the 4th German edition: *On the Sensations of Tone as a Physiological Basis for the Theory of Music* (1877). Reprinted by Dover Publications, 1954.

**[3]** Galilei, G. (1638). *Discorsi e dimostrazioni matematiche intorno a due nuove scienze*. Leiden: Elzevir. English translation: *Dialogues Concerning Two New Sciences*.

**[4]** Mersenne, M. (1636). *Harmonie universelle, contenant la theorie et la pratique de la musique*. Paris: Sebastien Cramoisy.

**[5]** Boyle, R. (1660). *New Experiments Physico-Mechanicall, Touching the Spring of the Air, and its Effects*. Oxford: H. Hall. Experiment 27 concerns sound in the evacuated receiver.

**[6]** Newton, I. (1687). *Philosophiae Naturalis Principia Mathematica*. London: Royal Society.

**[7]** Fourier, J. B. J. (1822). *Theorie analytique de la chaleur*. Paris: Firmin Didot. English translation: *The Analytical Theory of Heat*, Cambridge University Press, 1878.

**[8]** Cooley, J. W., and Tukey, J. W. (1965). An algorithm for the machine calculation of complex Fourier series. *Mathematics of Computation*, 19(90), 297 to 301. DOI: 10.1090/S0025-5718-1965-0178586-1

**[9]** Taylor, F. W. (1907). On the art of cutting metals. *Transactions of the ASME*, 28, 31 to 350.

**[10]** Schmitz, T. L., and Smith, K. S. (2019). *Machining Dynamics: Frequency Response to Improved Productivity*, 2nd ed. Springer. DOI: 10.1007/978-3-319-93707-6

**[11]** Schmitz, T. L., and Gomez, M. F. *Machining Dynamics: Jupyter Notebook Edition* (in preparation).
